# 형태소 분석

In [2]:
import sys
from pathlib import Path

# 프로젝트 루트: notebooks/ 하위 어느 깊이에서 실행해도 동작
_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "code" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "code"))
        break
else:
    raise FileNotFoundError("code/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
from project_paths import DATA_CAFE_ONLY, DATA_INTEGRATED, CONFIG_STOPWORDS

# 파일 불러오기
import pandas as pd
import re

df = pd.read_csv(DATA_CAFE_ONLY / '의대증원_cafedata_preprocess.csv') 

print(f"네이버 카페 데이터 개수: {len(df)}")

df.head(10)

네이버 카페 데이터 개수: 3712


,title,doc,like,comment_cnt,comment_list,ch,date,section
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4
3,수시 6장 머리터지기 시작했어요,글쓰기 전 선검색 바랍니다 미취학글 과외구인글 아이디 추천글 댓글 달린 글 삭제시 ...,1,11,[{'comment_content': '전체내신나오고 아이의 생기부가 어느정도 파악...,cafe,2025-06-30,4
4,고3 수시 대학 라인 문제,고1 고3 필수 학종 합격 가이드 1 학생 학부모 여부 본인및 자녀의 학년 예 고3...,1,2,"[{'comment_content': '같은 고민 입니다 어렵네요', 'commen...",cafe,2025-06-30,4
5,약대도 증원소식이 있나요,약대도 증원된다만다 이야기가 있고 2026년 대비 전략 이런 글들이 많던데약대 내년...,0,1,[{'comment_content': '증원얘기는 따로 없는것 같던데 예전에 약대 ...,cafe,2025-06-30,4
6,수박먹고 대학간다 기본편 2026,이번 중위권 간담뢰 다녀와서 본격적으로 공부하려 책을 펼칩니다 아이 등급에 따라 여...,10,15,"[{'comment_content': '보고 계신 건 기본편인거죠 제목 미스매치',...",cafe,2025-06-30,4
7,재수생 학부모 계신가요 궁금한게 있는데,1 게시글과 덧글은 성의있고 예의바르게 작성하도록 합니다 게시글은 PC로 최소 7줄...,1,10,[{'comment_content': '에고 전직 삼수생으로써 6평 잘봤다고 풀어지...,cafe,2025-06-30,4
8,보건복지부장관 정은경 지명됐네요,정회원 등업대기 공간입니다 카페 취지에 맞지 않는 글은 임의 삭제 될 수 있습니다 ...,6,45,"[{'comment_content': '인사청문회에서 물어뜯기면 어떡하죠', 'co...",cafe,2025-06-29,4
9,목차,1 시계편 전쟁은 계산이다 2024년 2월 6일 의대증원 정책이 발표되었고 20일 ...,4,4,[{'comment_content': '오 손자병법이었나요 어려운 내용을 겪어낸 일...,cafe,2025-06-27,4


In [62]:
from kiwipiepy import Kiwi
from tqdm import tqdm


# 1. 인스턴스(일꾼) 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

title_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
title_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(df))) :
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(df['title'][i])

    # 형태소 분석 ( Kiwi는 tokenize를 사룔)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장
    pos_result = [(t.form, t.tag) for t in tokens]
    title_token_list.append(pos_result)

    # 명사 추출
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.tag) > 1]
    title_token_noun.append(nouns)

Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
100%|█████████████████████████████████████| 3712/3712 [00:02<00:00, 1740.22it/s]


In [5]:
print(title_token_list)

[[('2026', 'SN'), ('학년도', 'NNG'), ('의과', 'NNG'), ('대학', 'NNG'), ('수시', 'NNG'), ('모집', 'NNG'), ('교과', 'NNG'), ('일반', 'NNG'), ('전형', 'NNG'), ('3', 'SN')], [('추가', 'NNG'), ('의대', 'NNG'), ('증원', 'NNG'), ('이야기', 'NNG'), ('나오', 'VV'), ('던데', 'EF')], [('sky', 'SL'), ('질문', 'NNG')], [('수시', 'NNG'), ('6', 'SN'), ('장', 'NNG'), ('머리', 'NNG'), ('터지', 'VV'), ('기', 'ETN'), ('시작', 'NNG'), ('하', 'XSV'), ('었', 'EP'), ('어요', 'EF')], [('고', 'NNG'), ('3', 'SN'), ('수시', 'NNG'), ('대학', 'NNG'), ('라인', 'NNG'), ('문제', 'NNG')], [('약대', 'NNG'), ('도', 'JX'), ('증원', 'NNG'), ('소식', 'NNG'), ('이', 'JKS'), ('있', 'VA'), ('나요', 'EF')], [('수박', 'NNG'), ('먹', 'VV'), ('고', 'EC'), ('대학', 'NNG'), ('가', 'VV'), ('ᆫ다', 'EF'), ('기본', 'NNG'), ('편', 'NNB'), ('2026', 'SN')], [('재수', 'NNG'), ('생', 'XSN'), ('학부모', 'NNG'), ('계시', 'VV'), ('ᆫ가요', 'EF'), ('궁금하', 'VA'), ('ᆫ', 'ETM'), ('것', 'NNB'), ('이', 'JKS'), ('있', 'VV'), ('는데', 'EC')], [('보건', 'NNG'), ('복지', 'NNG'), ('부', 'NNG'), ('장관', 'NNG'), ('정은경', 'NNP'), ('지명', 'NNG'), ('되', 'XSV

In [63]:
# 본문 명사 추출

# 1. 인스턴스(일꾼) 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

document_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
document_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(df))) :
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(df['doc'][i])

    # 형태소 분석 ( Kiwi는 tokenize를 사룔)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장
    pos_result = [(t.form, t.tag) for t in tokens]
    document_token_list.append(pos_result)

    # 명사 추출
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.tag) > 1]
    document_token_noun.append(nouns) 

Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
100%|███████████████████████████████████████| 3712/3712 [00:44<00:00, 82.68it/s]


In [57]:
print(document_token_noun[:10])

[['인제대', '작년', '내신', '성적', '반영', '때', '국영수', '과학', '상위', '과목', '반영', '학교', '내신', '반영', '방식', '특이', '학교', '직접', '비교', '대학', '공지', '작년', '기준', '컷', '인제대', '식', '등급', '만점', '기준', '경우', '국영수', '과', '과목', '반영', '대학', '일반', '등급', '정도', '해당', '국영', '수사과', '과목', '반영', '대학', '일반', '등급', '정도', '해당', '내신', '반대', '말', '올해', '국영수', '과', '과목', '정상', '반영', '수치', '내신', '컷', '등급', '정도', '상황', '의대', '정원', '낙관', '인제대', '면접', '때', '통과', '이후', '면접', '당락', '때', '반면', '해', '사실', '내신', '순', '줄', '중구난방', '예전', '면접', '최근', '면접', '때', '배수', '이후', '면접', '당락', '때', '인제대', '기본', '면접', '중요', '대학', '인하대', '경우', '작년', '정원', '와중', '작년', '등급', '탐구', '평균', '소수점', '최저', '학력', '등급', '올해', '합', '탐구', '소수점', '절', '사', '정원', '정원', '최저', '상황', '컷', '작년', '컷', '수능', '애매', '교과', '성적', '학생', '합', '합', '안전빵', '위', '합', '인하대', '합', '관동대', '건양', '대', '경상대', '합', '조선대', '강원대', '정도', '최저', '충족', '전제', '내신', '안전', '합격', '곳', '정도', '인하대', '확률', '등급', '사이', '입', '결', '방어', '인하대', '작년', '정시', '과학', '가산점', '만점', '감산점', '희', '방식', '작정', '입'

In [61]:
import ast
from tqdm import tqdm

comment_token_list = [] # 형태소 전체 결과 (단어, 태그) 리스트
comment_token_noun = [] # 2글자 이상 명사만 담는 리스트

for i in tqdm(range(len(df))):
    raw_comments = df['comment_list'][i]
    
    # 해당 게시글의 모든 댓글 결과를 합칠 임시 리스트
    post_full_tokens = []
    post_nouns_only_noun = []
    post_nouns_only_verb = []
    post_nouns_only_adj = []
    
    # 1. 데이터 타입 방어 (문자열일 경우 리스트로 변환)
    if isinstance(raw_comments, str):
        try:
            comments = ast.literal_eval(raw_comments)
        except:
            comments = []
    else:
        comments = raw_comments
        
    # 2. 분석 시작
    if isinstance(comments, list):
        for c in comments:
            if isinstance(c, dict):
                # 키 이름 확인 (comment_content)
                text = str(c.get('comment_content', ''))
                
                if text.strip():
                    tokens = kiwi.tokenize(text)
                    
                    for t in tokens:
                        # (1) 전체 결과 저장 (단어, 태그) 튜플 형태
                        post_full_tokens.append((t.form, t.tag))
                        
                        # (2) 명사 필터링 (2글자 이상 NNG, NNP)
                        if t.tag in ['NNG', 'NNP'] and len(t.form) > 1:
                            post_nouns_only_noun.append(t.form)
    
    # 게시글 단위로 최종 리스트에 추가
    comment_token_list.append(post_full_tokens)
    comment_token_noun.append(post_nouns_only_noun)

100%|███████████████████████████████████████| 3712/3712 [01:21<00:00, 45.27it/s]


# 불용어 처리(stopwords-ko.txt 기반)

In [31]:
import os
current_directory = os.getcwd()
print("현재 디렉토리:", current_directory)

현재 디렉토리: /Users/hyerimchoi/DJD/datacrolling/의대증원_중간프로젝트


In [32]:
# 불용어 사전 기반 불용어 리스트 정리
f = open(CONFIG_STOPWORDS / 'stopwords-ko.txt', 'r', encoding="UTF-8")
st = f.readlines()
f.close()

stw = []
for i in range(0, len(st)):
    stw.append(st[i].rstrip('\n'))

stw

['!',
 '"',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '...',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ';',
 '<',
 '=',
 '>',
 '?',
 '@',
 '\\',
 '^',
 '_',
 '`',
 '|',
 '~',
 '·',
 '—',
 '——',
 '‘',
 '’',
 '“',
 '”',
 '…',
 '、',
 '。',
 '〈',
 '〉',
 '《',
 '》',
 '가',
 '가까스로',
 '가령',
 '각',
 '각각',
 '각자',
 '각종',
 '갖고말하자면',
 '같다',
 '같이',
 '개의치않고',
 '거니와',
 '거바',
 '거의',
 '것',
 '것과 같이',
 '것들',
 '게다가',
 '게우다',
 '겨우',
 '견지에서',
 '결과에 이르다',
 '결국',
 '결론을 낼 수 있다',
 '겸사겸사',
 '고려하면',
 '고로',
 '곧',
 '공동으로',
 '과',
 '과연',
 '관계가 있다',
 '관계없이',
 '관련이 있다',
 '관하여',
 '관한',
 '관해서는',
 '구',
 '구체적으로',
 '구토하다',
 '그',
 '그들',
 '그때',
 '그래',
 '그래도',
 '그래서',
 '그러나',
 '그러니',
 '그러니까',
 '그러면',
 '그러므로',
 '그러한즉',
 '그런 까닭에',
 '그런데',
 '그런즉',
 '그럼',
 '그럼에도 불구하고',
 '그렇게 함으로써',
 '그렇지',
 '그렇지 않다면',
 '그렇지 않으면',
 '그렇지만',
 '그렇지않으면',
 '그리고',
 '그리하여',
 '그만이다',
 '그에 따르는',
 '그위에',
 '그저',
 '그중에서',
 '그치지 않다',
 '근거로',
 '근거하여',
 '기대여',
 '기점으로',
 '기준으로',
 '기타',
 '까닭으로',
 '까악',
 '까지',
 '까지 미치다',
 '까지도

In [33]:
# 각 문서의 제목과 본문 또는 댓글등에서 불용어 제거

for word in stw:
    for i in range(0, len(title_token_noun)) :
        # 리스트에 불용어가 있을 경우 제거
        while word in title_token_noun[i] :
            title_token_noun[i].remove(word)
    for j in range(0, len(document_token_noun)) : 
        while word in document_token_noun[j] :
            document_token_noun[j].remove(word)
    for l in range(0, len(comment_token_noun)) : 
        while word in comment_token_noun[l] :
            comment_token_noun[l].remove(word)

In [34]:
# 문서파일 docs로 적용하여 각각의 불용어 제거
df['title_token_list_pos'] = title_token_list # 형태소와 품사 리스트
df['title_token_noun'] = title_token_noun # 명사 리스트

df['document_token_list_pos'] = document_token_list
df['document_token_noun'] = document_token_noun

df['comment_token_list_pos'] = comment_token_list
df['comment_token_noun'] = comment_token_noun

In [37]:
# 불용어 처리 후 최종 파일 저장 및 불러오기
import pickle

f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press.pkl", 'wb')
pickle.dump(df, f)
f.close()

In [43]:
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press.pkl", 'rb')
data = pickle.load(f)
f.close()
data = data.drop(columns=['comment_nouns'])
data

,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_list_pos,title_token_noun,document_token_list_pos,document_token_noun,comment_token_noun,comment_token_list_pos
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4,"[(2026, SN), (학년도, NNG), (의과, NNG), (대학, NNG),...","[학년도, 의과, 대학, 수시, 모집, 교과, 일반, 전형]","[(인제대, NNP), (는, JX), (작년, NNG), (내신, NNG), (성...","[인제대, 작년, 내신, 성적, 반영, 국영수, 과학, 상위, 과목, 반영, 학교,...","[한눈, 전체, 내용, 부연, 설명, 감사, 객관, 인칭, 시점, 보믄, 파라미터,...","[(잘, MAG), (읽, VV), (어, EC), (보, VX), (었, EP),..."
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4,"[(추가, NNG), (의대, NNG), (증원, NNG), (이야기, NNG), ...","[추가, 의대, 증원, 이야기]","[(최근, NNG), (에, JKB), (뉴스, NNG), (보, VV), (니까,...","[최근, 뉴스, 의대, 추가, 증원, 사실, 검색, 추천, 강, 생물, 노용관, 이...","[올해, 유예, 취소, 유예, 회의, 내년, 증원, 결정]","[(올해, NNG), (는, JX), (일단, MAG), (유예, NNG), (되,..."
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4,"[(sky, SL), (질문, NNG)]",[질문],"[(언미생사, NNP), (all, SL), (백분위, NNG), (96, SN),...","[언미생사, 백분위, 인, 영어, 고정, 연고, 유리, 올해, 재수, 의대, 증원,...","[기준, 수능, 생각, 스카이, 유리, 생각, 이유, 성적, 유리, 수미, 잡이, ...","[(뭐, NP), (가, JKS), (기준, NNG), (이, VCP), (ᆫ지, ..."
3,수시 6장 머리터지기 시작했어요,글쓰기 전 선검색 바랍니다 미취학글 과외구인글 아이디 추천글 댓글 달린 글 삭제시 ...,1,11,[{'comment_content': '전체내신나오고 아이의 생기부가 어느정도 파악...,cafe,2025-06-30,4,"[(수시, NNG), (6, SN), (장, NNG), (머리, NNG), (터지,...","[수시, 장, 머리, 시작]","[(글, NNG), (쓰, VV), (기, ETN), (전, NNG), (선, NN...","[글, 전, 선, 검색, 취학, 글, 과외, 구인, 글, 아이디, 추천, 글, 댓글...","[전체, 내신, 생기, 정도, 파악, 객관, 현재, 위치, 성향, 종합, 교과, 논...","[(전체, NNG), (내신, NNG), (나오, VV), (고, EC), (아이,..."
4,고3 수시 대학 라인 문제,고1 고3 필수 학종 합격 가이드 1 학생 학부모 여부 본인및 자녀의 학년 예 고3...,1,2,"[{'comment_content': '같은 고민 입니다 어렵네요', 'commen...",cafe,2025-06-30,4,"[(고, NNG), (3, SN), (수시, NNG), (대학, NNG), (라인,...","[고, 수시, 대학, 라인, 문제]","[(고, NNG), (1, SN), (고, NNG), (3, SN), (필수, NN...","[고, 고, 필수, 학종, 합격, 가이드, 학생, 학부모, 본인, 자녀, 학년, 고...","[고민, 안녕, 회원님유니브클래스, 카페, 회원, 축하, 처음, 유니브클래스, 카페...","[(같, VA), (은, ETM), (고민, NNG), (이, VCP), (ᆸ니다,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3707,서울대 수시 합격생 11 가 미등록,1 정치 및 종교 저작권 위반 타학원 및 관계자에 대한 지나친 표현의 글은 작성금지...,0,12,"[{'comment_content': '다들 의대가려고 그런가바요', 'commen...",cafe,2024-01-02,1,"[(서울대, NNP), (수시, NNG), (합격, NNG), (생, XSN), (...","[서울대, 수시, 합격, 등록]","[(1, SN), (정치, NNG), (및, MAG), (종교, NNG), (저작,...","[정치, 종교, 저작, 위반, 학원, 관계자, 표현, 글, 작성, 금지, 사이트, ...","[의대, 의대, 영향, 의대, 생각, 자연, 학생, 의대, 중복, 지원, 경우, 생...","[(다, NNG), (들, XSN), (의대, NNG), (가, VV), (려고, ..."
3708,의대 증원 관련 기사입니다,단독 이과 수험생 62 교차지원 고려 의대증원 노린 반수 늘 듯 동아일보 단독 이과...,0,4,"[{'comment_content': '점점 의대중심의 세상이 되어가네요', 'co...",cafe,2024-01-01,1,"[(의대, NNG), (증원, NNG), (관련, NNG), (기사, NNG), (...","[의대, 증원, 관련, 기사]","[(단독, NNG), (이과, NNG), (수험, NNG), (생, XSN), (6...","[단독, 이과, 수험, 교차, 지원, 고려, 의대, 증원, 반수, 동아일보, 단독,...","[의대, 중심, 세상, 나중, 의대, 취급, 다양, 직업, 존재, 의대, 의대, 열...","[(점점, MAG), (의대, NNG), (중심, NNG), (의, JKG), (세..."
3709,부끄러운 얘기지만 치과는 여러 곳 가보세요,부끄러운 얘기지만 치과는 여러 곳 가보세요 치과계 폭리 다룬 책 펴낸 치과의사 김광...,6,5,[{'comment_content': '20 30년 정도 오래된 치과에 가는 것을 ...,cafe,2024-01-01,1,"[(부끄럽, VA-I), (은, ETM), (얘기, NNG), (이, VCP), (...","[얘기, 치과, 곳]","[(부끄럽, VA-I), (은, ETM), (얘기, NNG), (이, VCP), (...","[얘기, 치과, 곳, 치과, 폭리, 책, 치과, 의사, 김광수, 치과, 폭리, 지적...","[정도, 치과, 추천, 조언, 대형, 치과, 상업, 자유, 갑진, 새해, 시작, 새...","[(20, SN), (30, SN), (년, NNB), (정도, NNG), (오래,..."
3710,2024년신년사 서울아산병원장 청라 개원 관련 여러번 언급,뉴스 행사2024년 신년사 서울아산병원장 박승일2024 01 01 2024년 갑진년...,54,18,[{'comment_content': '중입자치료기도입은 서울에 한다는걸까요 청라에...,cafe,2024-01-01,1,"[(2024, SN), (년, NNB), (신년사, NNG), (서울아산병원, NN...","[신년사, 서울아산병원, 장, 청라, 개원, 관련, 언급]","[(뉴스, NNG), (행사, NNG), (2024, SN), (년, NNB), (...","[뉴스, 행사, 신년사, 서울아산병원, 장, 박승일, 갑진, 새해, 서울아산병원, ...","[중입자, 치료기, 도입, 서울, 청라, 도입, 변화, 혁신, 미래, 청라, 도입,...","[(중입자, NN

In [65]:
data_drop_list = data.drop(columns=['title_token_list_pos', 'document_token_list_pos', 'comment_token_list_pos'])
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl", 'wb')
pickle.dump(data_drop_list, f)
f.close()

In [67]:
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl", 'rb')
d = pickle.load(f)
f.close()
d

,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_noun,document_token_noun,comment_token_noun
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4,"[학년도, 의과, 대학, 수시, 모집, 교과, 일반, 전형]","[인제대, 작년, 내신, 성적, 반영, 국영수, 과학, 상위, 과목, 반영, 학교,...","[한눈, 전체, 내용, 부연, 설명, 감사, 객관, 인칭, 시점, 보믄, 파라미터,..."
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4,"[추가, 의대, 증원, 이야기]","[최근, 뉴스, 의대, 추가, 증원, 사실, 검색, 추천, 강, 생물, 노용관, 이...","[올해, 유예, 취소, 유예, 회의, 내년, 증원, 결정]"
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4,[질문],"[언미생사, 백분위, 인, 영어, 고정, 연고, 유리, 올해, 재수, 의대, 증원,...","[기준, 수능, 생각, 스카이, 유리, 생각, 이유, 성적, 유리, 수미, 잡이, ..."
3,수시 6장 머리터지기 시작했어요,글쓰기 전 선검색 바랍니다 미취학글 과외구인글 아이디 추천글 댓글 달린 글 삭제시 ...,1,11,[{'comment_content': '전체내신나오고 아이의 생기부가 어느정도 파악...,cafe,2025-06-30,4,"[수시, 장, 머리, 시작]","[글, 전, 선, 검색, 취학, 글, 과외, 구인, 글, 아이디, 추천, 글, 댓글...","[전체, 내신, 생기, 정도, 파악, 객관, 현재, 위치, 성향, 종합, 교과, 논..."
4,고3 수시 대학 라인 문제,고1 고3 필수 학종 합격 가이드 1 학생 학부모 여부 본인및 자녀의 학년 예 고3...,1,2,"[{'comment_content': '같은 고민 입니다 어렵네요', 'commen...",cafe,2025-06-30,4,"[고, 수시, 대학, 라인, 문제]","[고, 고, 필수, 학종, 합격, 가이드, 학생, 학부모, 본인, 자녀, 학년, 고...","[고민, 안녕, 회원님유니브클래스, 카페, 회원, 축하, 처음, 유니브클래스, 카페..."
...,...,...,...,...,...,...,...,...,...,...,...
3707,서울대 수시 합격생 11 가 미등록,1 정치 및 종교 저작권 위반 타학원 및 관계자에 대한 지나친 표현의 글은 작성금지...,0,12,"[{'comment_content': '다들 의대가려고 그런가바요', 'commen...",cafe,2024-01-02,1,"[서울대, 수시, 합격, 등록]","[정치, 종교, 저작, 위반, 학원, 관계자, 표현, 글, 작성, 금지, 사이트, ...","[의대, 의대, 영향, 의대, 생각, 자연, 학생, 의대, 중복, 지원, 경우, 생..."
3708,의대 증원 관련 기사입니다,단독 이과 수험생 62 교차지원 고려 의대증원 노린 반수 늘 듯 동아일보 단독 이과...,0,4,"[{'comment_content': '점점 의대중심의 세상이 되어가네요', 'co...",cafe,2024-01-01,1,"[의대, 증원, 관련, 기사]","[단독, 이과, 수험, 교차, 지원, 고려, 의대, 증원, 반수, 동아일보, 단독,...","[의대, 중심, 세상, 나중, 의대, 취급, 다양, 직업, 존재, 의대, 의대, 열..."
3709,부끄러운 얘기지만 치과는 여러 곳 가보세요,부끄러운 얘기지만 치과는 여러 곳 가보세요 치과계 폭리 다룬 책 펴낸 치과의사 김광...,6,5,[{'comment_content': '20 30년 정도 오래된 치과에 가는 것을 ...,cafe,2024-01-01,1,"[얘기, 치과, 곳]","[얘기, 치과, 곳, 치과, 폭리, 책, 치과, 의사, 김광수, 치과, 폭리, 지적...","[정도, 치과, 추천, 조언, 대형, 치과, 상업, 자유, 갑진, 새해, 시작, 새..."
3710,2024년신년사 서울아산병원장 청라 개원 관련 여러번 언급,뉴스 행사2024년 신년사 서울아산병원장 박승일2024 01 01 2024년 갑진년...,54,18,[{'comment_content': '중입자치료기도입은 서울에 한다는걸까요 청라에...,cafe,2024-01-01,1,"[신년사, 서울아산병원, 장, 청라, 개원, 관련, 언급]","[뉴스, 행사, 신년사, 서울아산병원, 장, 박승일, 갑진, 새해, 서울아산병원, ...","[중입자, 치료기, 도입, 서울, 청라, 도입, 변화, 혁신, 미래, 청라, 도입,..."
